# MK7 Native MoE — One-click Kaggle / Colab
Run All. This notebook clones/updates the experiment, verifies CUDA, runs the smoke gate, then runs the 30M research gate. It stops immediately on the first failure.

In [ ]:
%%bash
set -euo pipefail
ROOT=/kaggle/working/notebooks-fine-tuning
if [ ! -d /kaggle/working ]; then ROOT=$PWD/notebooks-fine-tuning; fi
if [ -d "$ROOT/.git" ]; then
  git -C "$ROOT" fetch origin exp/mk7-native-moe-streaming-v0
  git -C "$ROOT" checkout exp/mk7-native-moe-streaming-v0
  git -C "$ROOT" reset --hard origin/exp/mk7-native-moe-streaming-v0
else
  git clone --depth 1 -b exp/mk7-native-moe-streaming-v0 https://github.com/saskw2010/notebooks-fine-tuning.git "$ROOT"
fi
cd "$ROOT/experiments/mk7_native_moe"
python - <<'PY'
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA GPU is not enabled'
print('gpu:', torch.cuda.get_device_name(0))
print('vram_gb:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
PY
echo '=== SMOKE GATE ==='
python train_lm.py --preset smoke --steps 20 --batch-size 4
echo '=== RESEARCH 30M GATE ==='
python train_lm.py --preset research-30m --steps 100 --batch-size 2
echo '=== MK7 RUN COMPLETE ==='
